# Lab 6A: Structured prompts
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 6 · about 30 minutes**

**Objectives**
- Compare a messy prompt with a structured one
- Use examples to fix format drift
- Test long-document placement: document first versus question first
- Record measured differences

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

In [ ]:
POLICY = """TELECOM SUPPORT POLICY (fictional)
[P1] Duplicate charges are refunded in full within 7 working days.
[P2] Devices may be returned within 30 days of delivery if undamaged.
[P3] Outage credits: 1 day of service credit for every 24 hours of confirmed outage.
[P4] Plan downgrades take effect at the start of the next billing cycle.
[P5] Roaming packs cannot be refunded once activated.
[P6] Account ownership transfers require both parties to confirm by email."""
QUESTIONS = [("I was charged twice. When do I get my money back?", "P1"), ("Can I return a router I got 20 days ago?", "P2"),
             ("Our area was down for 48 hours. What credit do I get?", "P3"), ("If I downgrade today, when does it apply?", "P4"),
             ("Do you offer student discounts?", None)]

## 1. Messy versus structured
Same policy, same questions. The checks: under 80 words, cites a policy section, and admits when the policy is silent.

In [ ]:
MESSY = "you are support and here is policy " + POLICY.replace("\n", " ") + " answer the customer short and mention which rule, if not covered say so. customer: {q}"
STRUCTURED_SYSTEM = "You are a telecom billing support agent. Answer only from the policy."
STRUCTURED_USER = """<policy>
{policy}
</policy>

<instructions>
Answer the question using only the policy. Cite the section ID in square brackets, for example [P2].
If the policy does not cover the question, reply exactly: "Our policy does not cover that. I can connect you with the team."
Keep the reply under 80 words.
</instructions>

<question>
{q}
</question>"""
def check(reply, section):
    return {"under 80 words": len(reply.split()) <= 80,
            "cites right section": (f"[{section}]" in reply) if section else True,
            "admits gap": ("does not cover" in reply.lower()) if section is None else True}
rows = []
for q, sec in QUESTIONS:
    m = text_of(client.messages.create(model=config.MODEL, max_tokens=300, messages=[{"role": "user", "content": MESSY.format(q=q)}]))
    s = text_of(client.messages.create(model=config.MODEL, max_tokens=300, system=STRUCTURED_SYSTEM,
                                       messages=[{"role": "user", "content": STRUCTURED_USER.format(policy=POLICY, q=q)}]))
    rows.append({"question": q[:40], **{f"messy: {k}": v for k, v in check(m, sec).items()}, **{f"structured: {k}": v for k, v in check(s, sec).items()}})
df = pd.DataFrame(rows); display(df)
print("messy passes all:", int(df.filter(like="messy").all(axis=1).sum()), "/", len(df), "| structured passes all:", int(df.filter(like="structured").all(axis=1).sum()), "/", len(df))

## 2. Examples fix format drift
Target format, exactly: `Category: <word> | Next step: <short action>`

In [ ]:
TICKETS = ["Router reboots every hour", "Charged a roaming fee at home", "No signal on the whole street", "Forgot my password", "Want a cheaper plan", "App crashes on bill page"]
ZERO = "Classify the ticket and give the next step. Format: Category: <word> | Next step: <short action>"
FEW = ZERO + """
<examples>
<example>Ticket: Internet down across our building
Category: outage | Next step: Check the outage map and open a network incident</example>
<example>Ticket: Bill shows a fee I don't recognize
Category: billing | Next step: Review the charge and explain or refund it</example>
</examples>
The examples show the format only; do not reuse their content."""
FORMAT = re.compile(r"^Category: [a-z]+ \| Next step: .{3,80}$")
for name, sys_prompt in (("zero-shot", ZERO), ("two examples", FEW)):
    outs = [text_of(client.messages.create(model=config.FAST_MODEL, max_tokens=60, system=sys_prompt,
                                           messages=[{"role": "user", "content": f"Ticket: {t}"}])).strip() for t in TICKETS]
    print(f"{name:13s} exact format: {sum(bool(FORMAT.match(o)) for o in outs)}/{len(outs)}  e.g. {outs[0]!r}")

## 3. Long-document placement
A long synthetic handbook with five planted facts at different depths. We ask about them with the document first, then with the question first.

In [ ]:
random.seed(5)
filler = [f"Clause {i}: Agents record the account ID, confirm the customer's issue, and log the outcome with reason code R{i:03d}." for i in range(1, 161)]
PLANTED = {40: ("Clause 40: Fiber installation appointments can be rescheduled up to 2 times free of charge.", "How many times can a fiber installation appointment be rescheduled for free?", "2"),
           80: ("Clause 80: Lost SIM replacement costs 5 USD for prepaid customers.", "What does a lost SIM replacement cost for prepaid customers?", "5"),
           110: ("Clause 110: Business plans include a 4-hour repair commitment.", "What repair commitment do business plans include, in hours?", "4"),
           135: ("Clause 135: Paper bills cost 2 USD per month.", "How much do paper bills cost per month in USD?", "2"),
           155: ("Clause 155: Number porting completes within 3 working days.", "Within how many working days does number porting complete?", "3")}
for pos, (txt, _, _) in PLANTED.items(): filler[pos - 1] = txt
HANDBOOK = "\n".join(filler)
print("handbook tokens:", client.messages.count_tokens(model=config.FAST_MODEL, messages=[{"role": "user", "content": HANDBOOK}]).input_tokens)
def ask_handbook(question, doc_first):
    doc = f"<handbook>\n{HANDBOOK}\n</handbook>"
    instr = "Answer with just the number."
    content = f"{doc}\n\n{instr}\n<question>{question}</question>" if doc_first else f"<question>{question}</question>\n{instr}\n\n{doc}"
    return text_of(client.messages.create(model=config.FAST_MODEL, max_tokens=20, messages=[{"role": "user", "content": content}])).strip()
place = []
for pos, (_, q, ans) in PLANTED.items():
    place.append({"fact at clause": pos, "doc first correct": ans in ask_handbook(q, True), "question first correct": ans in ask_handbook(q, False)})
pd.DataFrame(place)

Small tests like this can vary run to run. The guidance is still worth adopting as a default: long material first, the question last.

## You should now have
- [ ] A messy versus structured comparison
- [ ] Format compliance with and without examples
- [ ] A document placement experiment

## Check yourself
1. Which structural change made the biggest difference?
2. Why do the examples say they show format only?
3. How would you make the placement test more reliable?